# 05 — Modelo Preditivo de Resultado de Ações

**Objetivo:** Treinar um modelo de classificação para prever o resultado
de processos judiciais (PROCEDENTE / IMPROCEDENTE / EXTINTO).

**Features utilizadas:**
- Tribunal, grau, classe processual, órgão julgador (categóricas → encoded)
- Duração em dias, total de movimentos (numéricas)

**Modelos disponíveis:** Random Forest, Gradient Boosting, Logistic Regression

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datajud.ml import ModeloPreditivo

sns.set_theme(style='whitegrid')

## 1. Carregar dados

In [ ]:
df_processos = pd.read_parquet('../data/processed/processos_tjsp.parquet')
df_resultados = pd.read_parquet('../data/processed/resultados_tjsp.parquet')

print(f'Processos: {df_processos.shape}')
print(f'Resultados: {df_resultados.shape}')
print(f'Distribuição de resultados:\n{df_resultados["resultado"].value_counts()}')

## 2. Preparar dados e treinar

In [ ]:
modelo = ModeloPreditivo(algoritmo='random_forest', n_estimators=200)

modelo.preparar_dados(
    df_processos,
    df_resultados,
    test_size=0.2,
)

modelo.treinar()

## 3. Avaliação

In [ ]:
resultado_avaliacao = modelo.avaliar()

## 4. Importância das features

In [ ]:
df_importancia = modelo.feature_importance(top_n=15)
df_importancia

## 5. Comparação de algoritmos

In [ ]:
from sklearn.metrics import f1_score

algoritmos = ['random_forest', 'gradient_boosting', 'logistic_regression']
resultados_comp = []

for alg in algoritmos:
    m = ModeloPreditivo(algoritmo=alg)
    m.preparar_dados(df_processos, df_resultados)
    m.treinar()
    y_pred = m.modelo.predict(m.X_test)
    f1 = f1_score(m.y_test, y_pred, average='weighted')
    resultados_comp.append({'algoritmo': alg, 'f1_weighted': f1})
    print(f'{alg}: F1={f1:.3f}')

df_comp = pd.DataFrame(resultados_comp)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=df_comp, x='algoritmo', y='f1_weighted', palette='Set2', ax=ax)
ax.set_title('Comparação de Algoritmos — F1 Weighted')
ax.set_ylim(0, 1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 6. Salvar o melhor modelo

In [ ]:
import pickle

with open('../data/processed/modelo_resultado.pkl', 'wb') as f:
    pickle.dump(modelo, f)

print('Modelo salvo em data/processed/modelo_resultado.pkl')